# Phase 2. RFM 세그멘테이션

**목표:** 2024년 기준 고객 세분화 → 9개 Named Segment 도출 → CRM 전략 기반 수립

## 왜 9개인가? (데이터 기반 근거)

일반적으로 알려진 Klaviyo 11개 세그먼트를 맹목적으로 따르지 않고,
**카카오 선물하기 실제 구매 데이터를 먼저 분석한 뒤** 세그먼트 수를 결정했다.

| 지표 | 발견 | 결정 |
|---|---|---|
| F (빈도) | 구매자 80%가 연 1~3회에 집중, 6회+ = 2.7%에 불과 | 5단계 → **4단계**로 축소 |
| R (최근성) | 0~365일 고르게 분포 | 5단계 유지 |
| M (금액) | 넓게 분포, 구간별 의미 있음 | 5단계 유지 |

F를 4단계로 줄이면 R×F 조합이 20개로 줄어들고,
비즈니스 액션이 실질적으로 다른 9개 그룹이 도출된다.

**분석 흐름:**
1. Section 0: 환경 설정
2. Section 1: RFM 분포 탐색 (세그먼트 수 결정 근거)
3. Section 2: RFM 점수화
4. Section 3: 9개 세그먼트 명명 및 분포
5. Section 4: GMV 기여도 분석 (Pareto 검증)
6. Section 5: 세그먼트 이동 추적 (H1 vs H2 2024)
7. Section 6: 세그먼트 프로파일

## Section 0. 환경 설정

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('인증 완료')

: 

In [ ]:
!pip install -q koreanize-matplotlib db-dtypes
import koreanize_matplotlib

from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings("ignore")

PROJECT_ID = "ds-ysy"
client     = bigquery.Client(project=PROJECT_ID)

# charts 폴더 생성 (없으면 자동 생성)
os.makedirs("charts", exist_ok=True)

# Office-style 색상 팔레트
PPTX_PALETTE = ["#2E75B6", "#ED7D31", "#A9D18E", "#FFC000", "#5B9BD5",
                "#70AD47", "#FF0000", "#7030A0", "#00B0F0", "#92D050"]
BLUE_DARK  = "#2E75B6"
BLUE_LIGHT = "#BDD7EE"
RED        = "#C00000"
GRAY       = "#D9D9D9"

SEGMENT_ORDER = [
    "Champions", "Loyal Customers", "Can't Lose Them", "At Risk",
    "Potential Loyalists", "New Customers", "Casual", "Dormant", "Inactive"
]

print("환경 설정 완료 — charts 폴더 생성됨")

: 

## Section 1. RFM 분포 탐색 — 세그먼트 수 결정 근거

**왜 이걸 먼저 하나요?**

세그먼트 수를 임의로 정하지 않기 위해서예요.
데이터를 먼저 보고 → "이 서비스에서 의미 있는 구분이 몇 단계인가"를 결정합니다.

특히 **F(구매 빈도)** 분포가 핵심이에요.
카카오 선물하기는 연 1~4회가 자연스러운 구매 패턴이라 일반 이커머스와 달리 세분화 기준이 달라야 합니다.

In [ ]:
# RFM 원시값 계산
query_rfm_raw = """
SELECT
  u.user_id,
  DATE_DIFF(DATE '2024-12-31', MAX(DATE(o.created_at)), DAY) AS recency_days,
  COUNT(DISTINCT o.order_id)                                  AS frequency,
  COALESCE(SUM(o.total_amount_krw), 0)                        AS monetary
FROM `ds-ysy.kakao_gift.users` u
LEFT JOIN `ds-ysy.kakao_gift.orders` o
  ON u.user_id = o.sender_user_id
  AND DATE(o.created_at) BETWEEN '2024-01-01' AND '2024-12-31'
  AND o.order_status != 'refunded'
GROUP BY u.user_id
"""
df_raw = client.query(query_rfm_raw).to_dataframe()
df_raw['recency_days'] = df_raw['recency_days'].fillna(999).astype(int)
buyers = df_raw[df_raw['frequency'] > 0]
print(f'전체 유저: {len(df_raw):,}명')
print(f'구매자: {len(buyers):,}명 ({len(buyers)/len(df_raw)*100:.1f}%)')
print(f'비구매자: {len(df_raw)-len(buyers):,}명 ({(len(df_raw)-len(buyers))/len(df_raw)*100:.1f}%)')

In [ ]:
# R / F / M 분포 시각화 — 세그먼트 단계 수 결정 근거
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# F 분포
ax = axes[0]
f_dist = buyers['frequency'].value_counts().sort_index()
colors_f = [BLUE_DARK if i <= 5 else GRAY for i in f_dist.index]
ax.bar(f_dist.index, f_dist.values, color=colors_f, edgecolor='white')
ax.set_title('F 분포 (연간 구매 횟수)', fontsize=12, fontweight='bold')
ax.set_xlabel('구매 횟수')
ax.set_ylabel('유저 수')
for x, y in zip(f_dist.index, f_dist.values):
    ax.text(x, y + 100, f'{y/len(buyers)*100:.1f}%', ha='center', fontsize=8)
# 4단계 구분선
ax.axvline(x=3.5, color=RED, linestyle='--', linewidth=1.5, label='4단계 구분')
ax.legend(fontsize=9)
ax.text(0.62, 0.85, '6회+ = 2.7%\n→ 세분화 불필요', transform=ax.transAxes,
        fontsize=8, color=RED, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# R 분포
ax = axes[1]
r_bins   = [0, 30, 60, 90, 180, 365]
r_labels = ['0-30일', '31-60일', '61-90일', '91-180일', '181-365일']
r_cut = pd.cut(buyers['recency_days'], bins=r_bins, labels=r_labels)
r_dist = r_cut.value_counts().reindex(r_labels)
ax.bar(r_labels, r_dist.values, color=BLUE_DARK, alpha=0.8, edgecolor='white')
ax.set_title('R 분포 (마지막 구매 경과일)', fontsize=12, fontweight='bold')
ax.set_xlabel('경과 기간')
ax.set_xticklabels(r_labels, rotation=20, ha='right')
for i, v in enumerate(r_dist.values):
    ax.text(i, v + 100, f'{v/len(buyers)*100:.1f}%', ha='center', fontsize=8)
ax.text(0.05, 0.9, '고르게 분포\n→ 5단계 유지', transform=ax.transAxes,
        fontsize=8, color=BLUE_DARK, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# M 분포
ax = axes[2]
m_bins   = [0, 20000, 50000, 100000, 200000, 2000000]
m_labels = ['~2만', '2~5만', '5~10만', '10~20만', '20만+']
m_cut = pd.cut(buyers['monetary'], bins=m_bins, labels=m_labels)
m_dist = m_cut.value_counts().reindex(m_labels)
ax.bar(m_labels, m_dist.values, color=BLUE_DARK, alpha=0.8, edgecolor='white')
ax.set_title('M 분포 (연간 총 구매금액)', fontsize=12, fontweight='bold')
ax.set_xlabel('금액 구간')
for i, v in enumerate(m_dist.values):
    ax.text(i, v + 100, f'{v/len(buyers)*100:.1f}%', ha='center', fontsize=8)
ax.text(0.05, 0.9, '고르게 분포\n→ 5단계 유지', transform=ax.transAxes,
        fontsize=8, color=BLUE_DARK, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('RFM 분포 탐색 — 세그먼트 단계 수 결정 근거', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('charts/rfm_distribution_basis.png', dpi=150, bbox_inches='tight')
plt.show()

print('결론: F는 4단계, R과 M은 5단계 적용')

## Section 2. RFM 점수화

**설계 원칙:**
- **R → NTILE(5):** 최근에 구매할수록 5점, 오래될수록 1점
- **F → Custom 4단계:** 데이터 분포 기반 (1회=1점, 2회=2점, 3회=3점, 4회+=4점)
- **M → NTILE(5):** 많이 쓸수록 5점

In [ ]:
query_rfm_scored = """
WITH rfm_raw AS (
  SELECT
    u.user_id,
    COALESCE(DATE_DIFF(DATE '2024-12-31', MAX(DATE(o.created_at)), DAY), 999) AS recency_days,
    COUNT(DISTINCT o.order_id)                                                 AS frequency,
    COALESCE(SUM(o.total_amount_krw), 0)                                       AS monetary
  FROM `ds-ysy.kakao_gift.users` u
  LEFT JOIN `ds-ysy.kakao_gift.orders` o
    ON u.user_id = o.sender_user_id
    AND DATE(o.created_at) BETWEEN '2024-01-01' AND '2024-12-31'
    AND o.order_status != 'refunded'
  GROUP BY u.user_id
),
rfm_scored AS (
  SELECT
    user_id,
    recency_days,
    frequency,
    monetary,
    -- R: 최근일수록 높은 점수 (역순 NTILE)
    NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
    -- F: 데이터 분포 기반 커스텀 4단계
    -- 근거: 구매자 80%가 연 1~3회, 6회+ = 2.7%로 세분화 불필요
    CASE
      WHEN frequency >= 4 THEN 4
      WHEN frequency  = 3 THEN 3
      WHEN frequency  = 2 THEN 2
      WHEN frequency  = 1 THEN 1
      ELSE 0
    END AS f_score,
    -- M: 구매금액 높을수록 높은 점수
    NTILE(5) OVER (ORDER BY monetary) AS m_score
  FROM rfm_raw
)
SELECT
  user_id,
  recency_days,
  frequency,
  monetary,
  r_score,
  f_score,
  m_score,
  -- 9개 세그먼트 (카카오 선물하기 맞춤 설계)
  CASE
    WHEN frequency = 0                              THEN 'Inactive'
    WHEN r_score >= 4 AND f_score >= 3              THEN 'Champions'
    WHEN r_score >= 3 AND f_score >= 3              THEN 'Loyal Customers'
    WHEN r_score <= 2 AND f_score >= 3 AND m_score >= 4 THEN 'Cant Lose Them'
    WHEN r_score <= 2 AND f_score >= 3              THEN 'At Risk'
    WHEN r_score = 5  AND f_score = 1              THEN 'New Customers'
    WHEN r_score >= 4 AND f_score = 2              THEN 'Potential Loyalists'
    WHEN r_score >= 3 AND f_score <= 2              THEN 'Casual'
    WHEN r_score <= 2 AND f_score <= 2              THEN 'Dormant'
    ELSE 'Casual'
  END AS rfm_segment
FROM rfm_scored
"""

df_rfm = client.query(query_rfm_scored).to_dataframe()
df_rfm['rfm_segment'] = df_rfm['rfm_segment'].replace('Cant Lose Them', "Can't Lose Them")
print(f'RFM 점수화 완료: {len(df_rfm):,}명')
df_rfm.head()

## Section 3. 세그먼트 분포

**9개 세그먼트 정의 (카카오 선물하기 맞춤)**

| 세그먼트 | 조건 | 의미 | CRM 목표 |
|---|---|---|---|
| **Champions** | R≥4, F≥3 | 최근에도 자주 구매하는 VIP | Lock-in: 신상품 선공개, 전용 혜택 |
| **Loyal Customers** | R=3, F≥3 | 꾸준한 단골, 조금 뜸해지는 중 | Retention: 정기 쿠폰, 생일 이벤트 |
| **Can't Lose Them** | R≤2, F≥3, M≥4 | 고가치인데 이탈 중 — 최우선 관리 | 즉각 케어: 15% 할인 + 개인화 메시지 |
| **At Risk** | R≤2, F≥3 | 자주 왔는데 요즘 안 옴 | 재활성화: 10% 할인 쿠폰 |
| **New Customers** | R=5, F=1 | 최근 처음 구매한 신규 고객 | Onboarding: 웰컴 메시지 + 가이드 |
| **Potential Loyalists** | R≥4, F=2 | 최근 왔고 단골 될 가능성 있음 | Convert: 다음 기념일 리마인더 |
| **Casual** | R≥3, F≤2 | 특정 이벤트 때만 가끔 구매 | Nurture: 시즌 이벤트 알림 |
| **Dormant** | R≤2, F≤2 | 오래 전 저빈도, 재활성화 필요 | Win-back: 신상품 알림 |
| **Inactive** | F=0 | 가입만 하고 한 번도 구매 안 함 | 선택적: 첫 구매 유도 쿠폰 |

In [ ]:
# 세그먼트별 집계
seg_stats = df_rfm.groupby('rfm_segment').agg(
    user_count=('user_id', 'count'),
    avg_recency=('recency_days', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    total_gmv=('monetary', 'sum')
).reset_index()

total_users = seg_stats['user_count'].sum()
total_gmv   = seg_stats['total_gmv'].sum()
seg_stats['user_pct'] = seg_stats['user_count'] / total_users * 100
seg_stats['gmv_pct']  = seg_stats['total_gmv']  / total_gmv  * 100

seg_stats['order'] = seg_stats['rfm_segment'].map({s: i for i, s in enumerate(SEGMENT_ORDER)})
seg_stats = seg_stats.sort_values('order').reset_index(drop=True)

print('=== 세그먼트별 현황 ===')
print(seg_stats[['rfm_segment','user_count','user_pct','gmv_pct','avg_monetary','avg_frequency']]
      .to_string(index=False, float_format=lambda x: f'{x:.1f}'))

In [ ]:
# Treemap -- 유저 수 vs GMV 기여도
try:
    import squarify
except ImportError:
    import subprocess; subprocess.run(['pip', 'install', '-q', 'squarify'])
    import squarify

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
colors_map = {seg: PPTX_PALETTE[i] for i, seg in enumerate(SEGMENT_ORDER)}

for ax, val_col, pct_col, title in zip(
    axes,
    ['user_count', 'total_gmv'],
    ['user_pct', 'gmv_pct'],
    ['세그먼트별 유저 수', '세그먼트별 GMV 기여']):

    # size=0인 세그먼트 제외 (squarify ZeroDivisionError 방지)
    mask = seg_stats[val_col] > 0
    filtered = seg_stats[mask].copy()

    sizes  = filtered[val_col].values
    labels = [f"{row['rfm_segment']}\n{row['user_pct']:.1f}% 유저\n{row['gmv_pct']:.1f}% GMV"
              for _, row in filtered.iterrows()]
    colors = [colors_map.get(s, GRAY) for s in filtered['rfm_segment']]

    squarify.plot(sizes=sizes, label=labels, color=colors,
                  alpha=0.85, ax=ax, text_kwargs={'fontsize': 9})
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.axis('off')

plt.suptitle('RFM 9-Segment Treemap', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('charts/rfm_treemap.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4. GMV 기여도 — Pareto 검증

**왜 이 분석?**

마케팅 예산을 어디에 써야 하는지 결정하기 위해서예요.
만약 상위 30% 고객이 GMV의 70%를 만들어낸다면 → 그 고객 유지에 집중 투자해야 한다는 근거가 생깁니다.
이게 바로 **Pareto 법칙(80:20 법칙)** 검증이에요.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

x      = np.arange(len(seg_stats))
colors = [colors_map.get(s, GRAY) for s in seg_stats['rfm_segment']]
bars   = ax1.bar(x, seg_stats['gmv_pct'], color=colors, alpha=0.85, edgecolor='white', width=0.6)

ax1.set_ylabel('GMV 기여 비중 (%)', fontsize=11)
ax1.set_xticks(x)
ax1.set_xticklabels(seg_stats['rfm_segment'], rotation=30, ha='right', fontsize=10)
ax1.set_ylim(0, seg_stats['gmv_pct'].max() * 1.3)

for bar, (_, row) in zip(bars, seg_stats.iterrows()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{row['gmv_pct']:.1f}%", ha='center', va='bottom', fontsize=9)
    if row['gmv_pct'] > 3:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
                 f"유저\n{row['user_pct']:.1f}%", ha='center', va='center',
                 fontsize=8, color='white', fontweight='bold')

# 누적 GMV 선
ax2 = ax1.twinx()
cumulative = seg_stats['gmv_pct'].cumsum()
ax2.plot(x, cumulative, color=RED, linewidth=2, marker='o', markersize=5)
ax2.axhline(80, color=RED, linestyle='--', linewidth=1, alpha=0.5)
ax2.set_ylabel('누적 GMV (%)', fontsize=11)
ax2.set_ylim(0, 115)
ax2.text(len(seg_stats)-1, 82, '80% 기준선', fontsize=9, color=RED)

ax1.set_title('세그먼트별 GMV 기여도 및 Pareto 검증', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/rfm_gmv_pareto.png', dpi=150, bbox_inches='tight')
plt.show()

# 핵심 수치 출력
top2 = seg_stats[seg_stats['rfm_segment'].isin(['Champions', 'Loyal Customers'])]
print(f'Champions + Loyal Customers')
print(f'  유저 비중: {top2["user_pct"].sum():.1f}%')
print(f'  GMV 기여: {top2["gmv_pct"].sum():.1f}%')

## Section 5. 세그먼트 이동 추적 (H1 → H2 2024)

**왜 이 분석?**

RFM은 특정 시점의 스냅샷이에요. 고객은 계속 이동해요.

예를 들어:
- Champions였던 고객이 6개월 후 At Risk가 됐다 → **즉각 개입 필요**
- Casual이었던 고객이 Loyal Customers로 올라왔다 → **잘 하고 있다는 신호**

H1(1~6월) → H2(7~12월) 이동 패턴을 보면 CRM이 효과가 있었는지도 알 수 있어요.

In [ ]:
def compute_rfm_segments(client, start_date, end_date, ref_date_str):
    """지정 기간의 RFM 세그먼트 계산"""
    q = f"""
    WITH rfm_raw AS (
      SELECT
        u.user_id,
        COALESCE(DATE_DIFF(DATE '{ref_date_str}', MAX(DATE(o.created_at)), DAY), 999) AS recency_days,
        COUNT(DISTINCT o.order_id)                                                     AS frequency,
        COALESCE(SUM(o.total_amount_krw), 0)                                           AS monetary
      FROM `ds-ysy.kakao_gift.users` u
      LEFT JOIN `ds-ysy.kakao_gift.orders` o
        ON u.user_id = o.sender_user_id
        AND DATE(o.created_at) BETWEEN '{start_date}' AND '{end_date}'
        AND o.order_status != 'refunded'
      GROUP BY u.user_id
    ),
    rfm_scored AS (
      SELECT
        user_id, recency_days, frequency, monetary,
        NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
        CASE WHEN frequency>=4 THEN 4 WHEN frequency=3 THEN 3
             WHEN frequency=2 THEN 2 WHEN frequency=1 THEN 1 ELSE 0 END AS f_score,
        NTILE(5) OVER (ORDER BY monetary) AS m_score
      FROM rfm_raw
    )
    SELECT user_id,
      CASE
        WHEN frequency = 0                                   THEN 'Inactive'
        WHEN r_score >= 4 AND f_score >= 3                   THEN 'Champions'
        WHEN r_score >= 3 AND f_score >= 3                   THEN 'Loyal Customers'
        WHEN r_score <= 2 AND f_score >= 3 AND m_score >= 4  THEN 'Cant Lose Them'
        WHEN r_score <= 2 AND f_score >= 3                   THEN 'At Risk'
        WHEN r_score  = 5 AND f_score  = 1                   THEN 'New Customers'
        WHEN r_score >= 4 AND f_score  = 2                   THEN 'Potential Loyalists'
        WHEN r_score >= 3 AND f_score <= 2                   THEN 'Casual'
        WHEN r_score <= 2 AND f_score <= 2                   THEN 'Dormant'
        ELSE 'Casual'
      END AS rfm_segment
    FROM rfm_scored
    """
    df = client.query(q).to_dataframe()
    df['rfm_segment'] = df['rfm_segment'].replace('Cant Lose Them', "Can't Lose Them")
    return df

df_h1 = compute_rfm_segments(client, '2024-01-01', '2024-06-30', '2024-06-30')
df_h2 = compute_rfm_segments(client, '2024-07-01', '2024-12-31', '2024-12-31')

df_migration = df_h1.merge(df_h2, on='user_id', suffixes=('_h1', '_h2'))
migration_matrix = (df_migration
    .groupby(['rfm_segment_h1', 'rfm_segment_h2'])
    .size()
    .reset_index(name='user_count'))

print(f'이동 추적 완료. 상위 이동 패턴:')
print(migration_matrix.sort_values('user_count', ascending=False).head(10).to_string(index=False))

In [ ]:
# 세그먼트 이동 히트맵
common_segs = [s for s in SEGMENT_ORDER if
               s in df_migration['rfm_segment_h1'].values and
               s in df_migration['rfm_segment_h2'].values]

pivot = (migration_matrix
    .pivot(index='rfm_segment_h1', columns='rfm_segment_h2', values='user_count')
    .fillna(0)
    .reindex(index=common_segs, columns=common_segs, fill_value=0))

fig, ax = plt.subplots(figsize=(13, 10))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='이동 유저 수')

ax.set_xticks(range(len(common_segs)))
ax.set_xticklabels(common_segs, rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(len(common_segs)))
ax.set_yticklabels(common_segs, fontsize=9)
ax.set_xlabel('H2 2024 세그먼트 (이동 후)', fontsize=11)
ax.set_ylabel('H1 2024 세그먼트 (이동 전)', fontsize=11)
ax.set_title('세그먼트 이동 행렬 (H1 → H2 2024)\n대각선 = 유지 / 위쪽 삼각형 = 이탈',
             fontsize=13, fontweight='bold')

max_val = pivot.values.max()
for i in range(len(common_segs)):
    for j in range(len(common_segs)):
        val = int(pivot.values[i, j])
        if val > 0:
            color = 'white' if val > max_val * 0.6 else 'black'
            ax.text(j, i, f'{val:,}', ha='center', va='center', fontsize=7, color=color)

# 대각선 강조 (유지된 고객)
for i in range(len(common_segs)):
    ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1,
                                fill=False, edgecolor=BLUE_DARK, linewidth=2))

plt.tight_layout()
plt.savefig('charts/rfm_migration_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 6. 세그먼트 프로파일

**왜 이 분석?**

세그먼트를 나눴으면 이제 "각 그룹이 어떤 사람들인가"를 구체화해야 해요.
Champions가 주로 어떤 카테고리를 선물하는지, Dormant는 어떤 계기에 구매했는지 알아야
"어떤 상품을 추천하는 캠페인 메시지를 보낼 것인가"를 결정할 수 있어요.

In [ ]:
# 세그먼트 + 주문 데이터 JOIN
query_profile = """
SELECT
  o.sender_user_id AS user_id,
  o.category,
  o.gift_occasion,
  o.total_amount_krw
FROM `ds-ysy.kakao_gift.orders` o
WHERE DATE(o.created_at) BETWEEN '2024-01-01' AND '2024-12-31'
  AND o.order_status != 'refunded'
"""
df_orders = client.query(query_profile).to_dataframe()
df_profile = df_rfm[['user_id','rfm_segment','monetary','frequency']].merge(
    df_orders, on='user_id', how='left')

print(f'프로파일 데이터 준비 완료: {len(df_profile):,}행')

In [ ]:
# 세그먼트별 평균 GMV / 빈도 비교
seg_profile = df_rfm.groupby('rfm_segment').agg(
    avg_monetary=('monetary', 'mean'),
    avg_frequency=('frequency', 'mean'),
    user_count=('user_id', 'count')
).reset_index()
seg_profile['order'] = seg_profile['rfm_segment'].map({s: i for i, s in enumerate(SEGMENT_ORDER)})
seg_profile = seg_profile.dropna(subset=['order']).sort_values('order')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, unit, color, title in zip(
    axes,
    ['avg_monetary', 'avg_frequency'],
    ['천원', '회'],
    [BLUE_DARK, '#ED7D31'],
    ['세그먼트별 평균 연간 GMV', '세그먼트별 평균 구매 빈도']):

    divisor = 1000 if col == 'avg_monetary' else 1
    vals    = seg_profile[col] / divisor
    bars    = ax.barh(seg_profile['rfm_segment'], vals, color=color, alpha=0.8, edgecolor='white')
    ax.set_xlabel(f'평균 ({unit})', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    for bar in bars:
        w = bar.get_width()
        label = f'₩{w:.0f}K' if col == 'avg_monetary' else f'{w:.1f}회'
        ax.text(w + vals.max()*0.01, bar.get_y() + bar.get_height()/2,
                label, va='center', fontsize=9)

plt.suptitle('세그먼트 프로파일', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/rfm_segment_profile.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 주요 세그먼트별 Top 3 선호 카테고리
focus_segs = ['Champions', 'Loyal Customers', "Can't Lose Them", 'At Risk', 'Dormant']
print('=== 세그먼트별 Top 3 선호 카테고리 ===')
for seg in focus_segs:
    seg_data = df_profile[df_profile['rfm_segment'] == seg].dropna(subset=['category'])
    if len(seg_data) == 0:
        continue
    top_cats = (seg_data.groupby('category')['total_amount_krw']
                .sum().sort_values(ascending=False).head(3))
    total = top_cats.sum()
    cats_str = ', '.join([f"{cat}({v/total*100:.0f}%)" for cat, v in top_cats.items()])
    print(f'  {seg:20s}: {cats_str}')

## 인사이트 요약

### 세그먼트 수 결정 근거
카카오 선물하기 구매자의 80%가 연 1~3회 구매 → F를 4단계로 축소 → 데이터 기반 9개 세그먼트 도출
(Klaviyo 11개를 맹목적으로 따르지 않고, 서비스 특성에 맞게 재설계)

### CRM 우선순위

| 우선순위 | 세그먼트 | 핵심 액션 |
|---|---|---|
| 1순위 | **Can't Lose Them** | 고가치 이탈 직전 — 즉각 15% 할인 + 개인화 메시지 |
| 2순위 | **Champions** | VIP Lock-in — 신상품 선공개, 전용 혜택 |
| 3순위 | **At Risk** | 재활성화 — 10% 할인 쿠폰 |
| 4순위 | **Potential Loyalists** | 단골 전환 — 다음 기념일 리마인더 |
| 5순위 | **Dormant** | Win-back — 시즌 이벤트 알림 (비용 효율 고려) |